# Local linear regimes

"the relationship holds up to a point, then changes" — the profiler's
regime pass looks for exactly that. It fits a bounded model tree with
linear models in its leaves (an M5-style tree, after Quinlan; smoothing
after Wang & Witten), and reports the local equations only when they beat
a single global line on held-out rows. A second, narrower probe —
residual level shifts — checks whether the *residuals* of a global
linear model step up or down along the first date column.

## When it runs

The pass is a follow-up, not a census: numeric targets are shortlisted by
the predictability screen, and at most 4 targets (8 on `deep`) are
fitted, over at most 5,000 rows (10,000) with at most 8 numeric
predictors (10), to depth 3 (4). A 1/5 holdout partition — the stable
row-id hash shared with the rule evaluator — must contain at least 60
rows, or the target is skipped. Only numeric predictors participate; the
first date column, if any, is used for ordering and for the residual
probe.

## The model tree

Training rows are standardized per feature and target — so stopping rules
are independent of units, and reported equations are mapped back to
original units. Splits are chosen by standard-deviation reduction over
coarse quantile candidates followed by local refinement; each leaf's
linear model is fitted by a Householder QR decomposition, the numerically
tidy way to solve small least squares. Corrected-error pruning removes
branches that do not pay for themselves, and ancestor smoothing shrinks
leaf models toward their parents — the M5' treatment, which keeps
equations stable when segments are thin.

The finding reports what the data support and nothing else: the leaf
conditions, each leaf's intercept and coefficients, its support, the
holdout R², and the holdout R² of one global linear model on the same
rows. It is emitted only when holdout R² ≥ 0.7 and the tree reduces
error by at least 10% relative to the global line. The statement is
`dependencies/predictive` with form `piecewise_linear`.

## Residual level shifts

For the global linear model fitted to each regime target, the pass orders
the *training* residuals by the first date column and scans for one
breakpoint — a centered cumulative-sum scan that finds the single split
minimizing within-segment squared residual means. A fixed correction
(before-breakpoint rows shift by one offset, after-breakpoint rows by
another) is then evaluated on the holdout: it must reduce holdout squared
error, and the reduction is the effect the finding reports.

The statement is `changes/residual_level`, and it carries its
references: the breakpoint date, both offsets, the error reduction, the
train/holdout sizes, and the reference model — the linear equation whose
residuals shifted, with its intercept and coefficients, so a reader can
reproduce the residuals from the columns. No p-value is attached: this is
a measured level change under a stated model, not an inferential claim.

In [ ]:
import numpy as np
import polars as pl

import onceover

rng = np.random.default_rng(41)
n = 6_000
x = rng.uniform(0, 10, n)
y = np.where(x < 5, x, 3 * x - 10) + rng.normal(0, 0.15, n)   # slope changes at x = 5

df = pl.DataFrame({"x": x, "y": y,
                    "noise": rng.random(n)})

report = onceover.profile(df)
piecewise = [f for f in report.findings
             if f["specification"].get("form") == "piecewise_linear"
             and f["subject"].get("target") == "y"]
print("piecewise findings for y:", len(piecewise))
for f in piecewise:
    print(f["claim"])
    for leaf in f["evidence"][0]["metrics"]["leaves"]:
        conds = " and ".join(f"{c['column']} {c['operator']} {c['threshold']:.3f}"
                             for c in leaf["conditions"])
        coefs = " + ".join(f"{v:.3f}·{k}" for k, v in leaf["coefficients"].items())
        print(f"   when {conds}:")
        print(f"      y ≈ {leaf['intercept']:.3f} + {coefs}  (n_train={leaf['n_train']}, n_holdout={leaf['n_holdout']})")

Reading the output: the tree splits coarsely first, then refines toward the
planted breakpoint — four local segments, each with its own equation in
original units and its train/holdout support. The coefficients tell the
story the fixture planted: near-unit slope below the change, near-three
above it. The holdout comparison — tree versus one global line on the same
rows — is the honesty mechanism: a tree that cannot beat the line by the
required margin is not reported, which is why most targets yield no
regime finding at all.